In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import sklearn

In [ ]:
# Load data
X_train = pd.read_parquet('../data/X_train.parquet')
y_train = pd.read_parquet('../data/y_train.parquet')


start = time.time()

# Historical baseline: ONE row per series, plain aggregation, no expanding
def compute_historical_baseline(series):
    grouped = series.groupby(level='id')['value']
    return pd.DataFrame({
        'mean': grouped.mean(),
        'median': grouped.median(),
        'std': grouped.std(),
        'skew': grouped.skew(),
        'kurt': grouped.apply(lambda x: x.kurtosis()),
        'min': grouped.min(),
        'max': grouped.max(),
        'autocorr': grouped.apply(lambda x: x.autocorr(lag=1))
    })

historical_series = X_train.query("period == 1")
historical_baseline = compute_historical_baseline(historical_series)


# Online expanding stats: ONE row per (id, timestep) using only data seen so far
def compute_expanding_stats(series):
    grouped = series.groupby(level='id')['value']
    return pd.DataFrame({
        'mean': grouped.expanding().mean(),
        'median': grouped.expanding().median(),
        'std': grouped.expanding().std(),
        'skew': grouped.expanding().skew(),
        'kurt': grouped.expanding().apply(lambda x: x.kurtosis()),
        'min': grouped.expanding().min(),
        'max': grouped.expanding().max(),
        'autocorr': grouped.expanding().apply(lambda x: x.autocorr(lag=1))
    })

online_series = X_train.query("period == 2")

online_expanding = compute_expanding_stats(online_series)
online_expanding = online_expanding.droplevel(0)

display(online_expanding.head()

display(historical_baseline.head())

/Users/amine/miniconda3/envs/structural_break/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:3015: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)


In [ ]:
merged = online_expanding.join(historical_baseline, on='id', lsuffix='_online', rsuffix='_hist')

metrics = ['mean', 'median', 'std', 'skew', 'kurt', 'min', 'max', 'autocorr']

# Compute delta between features
for m in metrics:
    merged[f'{m}_diff'] = merged[f'{m}_online'] - merged[f'{m}_hist']

# Keep only diff columns
diff_cols = [f'{m}_diff' for m in metrics]
merged = merged[diff_cols]

merged = merged.join(y_train)

merged.head()

mean_diff  median_diff  std_diff  skew_diff  kurt_diff  min_diff  \
id time                                                                     
0  1192   0.078301     0.080824       NaN        NaN        NaN  3.460292   
   1193  -0.419521    -0.416998 -0.296393        NaN        NaN  2.464647   
   1194  -1.193244    -0.914821  0.429184  -0.851354        NaN  0.641301   
   1195  -1.493399    -1.653081  0.312167   0.398376  -3.920450  0.641301   
   1196  -0.918921    -0.914821  0.714890   0.304321  -2.410219  0.641301   

         max_diff  autocorr_diff  target  
id time                                   
0  1192 -4.449895            NaN       0  
   1193 -4.449895            NaN       0  
   1194 -4.449895       1.015932       0  
   1195 -4.449895       0.659540       0  
   1196 -3.149204      -0.157124       0